In [3]:
# 0.1 Install necessary dependencies
!pip install requests

In [1]:
# 0.2 Import required libraries
import pandas as pd
import numpy as np
import time
import os
import json
import re
import docx
import glob
import requests

In [3]:
# 0.3 Set path variables
CSV_path = 'Cases per paper.csv'
publications_dir = 'preprocess_outputs'
extraction_dir = 'LLM_extracted_IPD'

pilot_path = os.path.join(publications_dir, "pilot_set.json")
development_path = os.path.join(publications_dir, "development_set.json")
validation_path = os.path.join(publications_dir, "validation_set.json")

In [5]:
# 0.4 Load the splited datasets from JSON files
with open(pilot_path, "r", encoding="utf-8") as f:
    pilot_data = json.load(f)
    
with open(development_path, "r", encoding="utf-8") as f:
    development_data = json.load(f)
    
with open(validation_path, "r", encoding="utf-8") as f:
    validation_data = json.load(f)

In [7]:
# 0.5 Load the cases-per-study CSV file and standardize the study name
cases_df = pd.read_csv(CSV_path)
cases_df["Study Name"] = cases_df["0"].astype(str)

In [9]:
# 1.1 Initialize the chat completion of OpenRouter
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
api_key = os.getenv("OPENROUTER_API_KEY")

In [11]:
# 1.2 Set the prompt for code testing
prompt_files = sorted(glob.glob("32 Prompts/*.docx"))
prompts = {}
for pf in prompt_files:
    doc = docx.Document(pf)
    text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    prompts[os.path.basename(pf).split(".")[0]] = text
print(prompts.keys())

dict_keys(['00000', '00001', '00010', '00011', '00100', '00101', '00110', '00111', '01000', '01001', '01010', '01011', '01100', '01101', '01110', '01111', '10000', '10001', '10010', '10011', '10100', '10101', '10110', '10111', '11000', '11001', '11010', '11011', '11100', '11101', '11110', '11111'])


In [33]:
# 1.3 Set the model to be tested
model = "openai/gpt-5-mini"

In [25]:
pilot_data.keys()

dict_keys([' Solís - Hasbun, 2016', 'Tarula ', 'Safadieh', 'Kurita - Mori, 2015 ', 'Kim, Kim - Lee, 2016'])

In [47]:
call_openrouter(model, prompts["00010"], pilot_data["Kim, Kim - Lee, 2016"])

'[\n  {\n    "Female": 1,\n    "Age": 3.0,\n    "SpeechDys": 1,\n    "Seizures": 1,\n    "MvmtDis": 1,\n    "Obtunded": 1,\n    "BrainstemDys": 1,\n    "BehCogImp": 1,\n    "ITU": 0,\n    "WorstMRSAcute": 4,\n    "EEGSlow": 1,\n    "EEGDeltaBrush": 1,\n    "EEGEpileptiform": 1,\n    "CSFPleo": 0,\n    "CSFWBCQuant": 2,\n    "AbnMRIBrain": 1,\n    "Tumour": 0,\n    "DaysToIT": 28,\n    "ITWithin30Days": 1,\n    "IT1stLineCombo": "CS+IVIG",\n    "IT2ndLineRTX": 1,\n    "IT2ndLineCYC": 0,\n    "IT2ndLineBort": 0,\n    "IT2ndLineToc": 0,\n    "ITMaintenanceMMF": 0,\n    "ITMaintenanceAZA": 0,\n    "ITMaintenanceMTX": 0,\n    "IT6mSteroid": 0,\n    "IT6mIVIG": 0,\n    "MonthsToFU": 12.0,\n    "FinalMRS": 4,\n    "Relapse": 0\n  }\n]'

In [15]:
# 2.1 Define a function to call the OpenRouter API
def call_openrouter(model, prompt, text):
    """
    Call the OpenRouter API to obtain model-generated responses.
    Parameters:
        model (str): The name of the model to use.
        prompt (str): The system prompt to define the model's behavior or role.
        text (str): The user's input text, which the model will respond to.
    Returns:
        str: The text content generated by the model.
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",              ### Request body is JSON
    }

    payload = {
        "model": model,
        "temperature": 0,                                ### Set temperature to 0 for reproducability
        "messages": [
            {"role": "system", "content": prompt},
            {"role": "user", "content": text}
        ]
    }
    
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()                               ### Parse returned JSON
    return data["choices"][0]["message"]["content"]

In [17]:
# 2.1 Define a function to extract and stucture IPD from single case reports 
def extract_case_report(model, prompt, study, text):
    """
    Extract IPD from a single case report.
    Parameters:
        model: The LLM instance used.
        prompt: The prompt text guiding extraction.
        study: Identifier of the study.
        text: The full text content of the single case report.
    Returns:
        A list containing the study name, extraction duration, and extracted patient features.
    """
    start_time = time.time()                                                                            ### Record start/end time for measuring duration

    output_text = call_openrouter(model, prompt, text)
    
    end_time = time.time()
    duration = end_time - start_time

    output_json = output_text.strip().removeprefix("```json").removesuffix("```").strip()               ### Clean up JSON code block markers
    data = json.loads(output_json)                                                                      ### Parse the JSON string to Python data structures                   
    
    if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):                          ### Handle case where output is a list with a dict or just a dict
        IPD = data[0]
    elif isinstance(data, dict):
        IPD = data
        
    Female = IPD.get("Female", None)
    Age = IPD.get("Age", None)
    SpeechDys = IPD.get("SpeechDys", None)
    Seizures = IPD.get("Seizures", None)
    MvmtDis = IPD.get("MvmtDis", None)
    Obtunded = IPD.get("Obtunded", None)
    BrainstemDys = IPD.get("BrainstemDys", None)
    BehCogImp = IPD.get("BehCogImp", None)
    ITU = IPD.get("ITU", None)
    WorstMRSAcute = IPD.get("WorstMRSAcute", None)
    EEGSlow = IPD.get("EEGSlow", None)
    EEGDeltaBrush = IPD.get("EEGDeltaBrush", None)
    EEGEpileptiform = IPD.get("EEGEpileptiform", None)
    CSFPleo = IPD.get("CSFPleo", None)
    CSFWBCQuant = IPD.get("CSFWBCQuant", None)
    AbnMRIBrain = IPD.get("ITU", None)
    Tumour = IPD.get("Tumour", None)
    DaysToIT = IPD.get("DaysToIT", None)
    ITWithin30Days = IPD.get("ITWithin30Days", None)
    IT1stLineCombo = IPD.get("IT1stLineCombo", None)
    IT2ndLineRTX = IPD.get("IT2ndLineRTX", None)
    IT2ndLineCYC = IPD.get("IT2ndLineCYC", None)
    IT2ndLineBort = IPD.get("IT2ndLineBort", None)
    IT2ndLineToc = IPD.get("IT2ndLineToc", None)
    ITMaintenanceMMF = IPD.get("ITMaintenanceMMF", None)
    ITMaintenanceAZA = IPD.get("ITMaintenanceAZA", None)
    ITMaintenanceMTX = IPD.get("ITMaintenanceMTX", None)
    IT6mSteroid = IPD.get("IT6mSteroid", None)
    IT6mIVIG = IPD.get("IT6mIVIG", None)
    MonthsToFU = IPD.get("MonthsToFU", None)
    FinalMRS = IPD.get("FinalMRS", None)
    Relapse = IPD.get("Relapse", None)
    
    output = [
        study, duration,
        Female, Age, SpeechDys, Seizures, MvmtDis, Obtunded, BrainstemDys, BehCogImp, ITU, WorstMRSAcute,
        EEGSlow, EEGDeltaBrush, EEGEpileptiform, CSFPleo, CSFWBCQuant, AbnMRIBrain, Tumour, DaysToIT, ITWithin30Days, IT1stLineCombo,
        IT2ndLineRTX, IT2ndLineCYC, IT2ndLineBort, IT2ndLineToc, ITMaintenanceMMF, ITMaintenanceAZA, ITMaintenanceMTX, IT6mSteroid, IT6mIVIG, MonthsToFU,
        FinalMRS, Relapse
    ]
    
    return output

In [19]:
# 2.2 Define a function to extract and stucture multiple entries of IPD from case series 
def extract_case_series(model, prompt, study, text):
    """
    Extract IPD from a case series.
    Parameters:
        model: The LLM instance used.
        prompt: The prompt text guiding extraction.
        study: Identifier of the study.
        text: The full text content of the single case report.
    Returns:
        A list of lists, each inner list representing extracted features for a single patient in the series.
    """
    start_time = time.time()

    output_text = call_openrouter(model, prompt, text)
       
    end_time = time.time()
    duration = end_time - start_time

    output_json = output_text.strip().removeprefix("```json").removesuffix("```").strip()
    data = json.loads(output_json)
    
    output = []                                                                     ### Define a list to contain lists of each series
    for IPD in data:                                                                ### Iterate over each patient's data in the series
        Female = IPD.get("Female", None)
        Age = IPD.get("Age", None)
        SpeechDys = IPD.get("SpeechDys", None)
        Seizures = IPD.get("Seizures", None)
        MvmtDis = IPD.get("MvmtDis", None)
        Obtunded = IPD.get("Obtunded", None)
        BrainstemDys = IPD.get("BrainstemDys", None)
        BehCogImp = IPD.get("BehCogImp", None)
        ITU = IPD.get("ITU", None)
        WorstMRSAcute = IPD.get("WorstMRSAcute", None)
        EEGSlow = IPD.get("EEGSlow", None)
        EEGDeltaBrush = IPD.get("EEGDeltaBrush", None)
        EEGEpileptiform = IPD.get("EEGEpileptiform", None)
        CSFPleo = IPD.get("CSFPleo", None)
        CSFWBCQuant = IPD.get("CSFWBCQuant", None)
        AbnMRIBrain = IPD.get("ITU", None)
        Tumour = IPD.get("Tumour", None)
        DaysToIT = IPD.get("DaysToIT", None)
        ITWithin30Days = IPD.get("ITWithin30Days", None)
        IT1stLineCombo = IPD.get("IT1stLineCombo", None)
        IT2ndLineRTX = IPD.get("IT2ndLineRTX", None)
        IT2ndLineCYC = IPD.get("IT2ndLineCYC", None)
        IT2ndLineBort = IPD.get("IT2ndLineBort", None)
        IT2ndLineToc = IPD.get("IT2ndLineToc", None)
        ITMaintenanceMMF = IPD.get("ITMaintenanceMMF", None)
        ITMaintenanceAZA = IPD.get("ITMaintenanceAZA", None)
        ITMaintenanceMTX = IPD.get("ITMaintenanceMTX", None)
        IT6mSteroid = IPD.get("IT6mSteroid", None)
        IT6mIVIG = IPD.get("IT6mIVIG", None)
        MonthsToFU = IPD.get("MonthsToFU", None)
        FinalMRS = IPD.get("FinalMRS", None)
        Relapse = IPD.get("Relapse", None)
    
        records = [
            study, duration,
            Female, Age, SpeechDys, Seizures, MvmtDis, Obtunded, BrainstemDys, BehCogImp, ITU, WorstMRSAcute,
            EEGSlow, EEGDeltaBrush, EEGEpileptiform, CSFPleo, CSFWBCQuant, AbnMRIBrain, Tumour, DaysToIT, ITWithin30Days, IT1stLineCombo,
            IT2ndLineRTX, IT2ndLineCYC, IT2ndLineBort, IT2ndLineToc, ITMaintenanceMMF, ITMaintenanceAZA, ITMaintenanceMTX, IT6mSteroid, IT6mIVIG, MonthsToFU,
            FinalMRS, Relapse
        ]
        output.append(records)
        
    return output

In [21]:
# 2.3 Define a function to batch process a dataset of study and texts
def batch_feature_extraction(model, prompt, dataset):
    """
    Extract IPD either as single case reports or case series depending on the number of cases.
    Parameters:
        model: The LLM instance used.
        prompt: The prompt text guiding extraction.
        dataset: Dictionary mapping study identifiers to their full text content.
    Returns:
        A pandas DataFrame containing all extracted features for all studies/cases.
    """
    records = []
    for study, text in dataset.items():
        n_cases = cases_df.loc[cases_df["Study Name"] == study, "count"].values[0]          ### Look up number of cases for this study
        
        if n_cases == 1:
            records.append(extract_case_report(model, prompt, study, text))
        else:
            series_records = extract_case_series(model, prompt, study, text)
            records.extend(series_records)
            
        time.sleep(5)                                                                       ### Pause 5 seconds between each request to avoid rate limiting.

    df = pd.DataFrame(records, columns=["Study", "Duration", 
                                        "Female", "Age", "SpeechDys", "Seizures", "MvmtDis", "Obtunded", "BrainstemDys", "BehCogImp", "ITU", "WorstMRSAcute",
                                        "EEGSlow", "EEGDeltaBrush", "EEGEpileptiform", "CSFPleo", "CSFWBCQuant", "AbnMRIBrain", "Tumour", "DaysToIT", "ITWithin30Days", "IT1stLineCombo",
                                        "IT2ndLineRTX", "IT2ndLineCYC", "IT2ndLineBort", "IT2ndLineToc", "ITMaintenanceMMF", "ITMaintenanceAZA", "ITMaintenanceMTX", "IT6mSteroid", "IT6mIVIG", "MonthsToFU",
                                        "FinalMRS", "Relapse"
                                       ])
    return df                                                                               ### Convert all records into a DataFrame

In [49]:
# 3.1 Run batch extraction on the pilot dataset
results_pilot = {}
for prompt_id, prompt in prompts.items():
    print(f"Running extraction with prompt {prompt_id} ...")
    records = []

    pilot_output = batch_feature_extraction(model, prompt, pilot_data)
    pilot_output_path = os.path.join(extraction_dir, f'pilot_output_{prompt_id}.csv')
    pilot_output.to_csv(pilot_output_path, index=False) 
    
    results_pilot[prompt_id] = pilot_output

Running extraction with prompt 00000 ...
Running extraction with prompt 00001 ...
Running extraction with prompt 00010 ...
Running extraction with prompt 00011 ...
Running extraction with prompt 00100 ...
Running extraction with prompt 00101 ...
Running extraction with prompt 00110 ...
Running extraction with prompt 00111 ...
Running extraction with prompt 01000 ...
Running extraction with prompt 01001 ...
Running extraction with prompt 01010 ...
Running extraction with prompt 01011 ...
Running extraction with prompt 01100 ...
Running extraction with prompt 01101 ...
Running extraction with prompt 01110 ...
Running extraction with prompt 01111 ...
Running extraction with prompt 10000 ...
Running extraction with prompt 10001 ...
Running extraction with prompt 10010 ...
Running extraction with prompt 10011 ...
Running extraction with prompt 10100 ...
Running extraction with prompt 10101 ...
Running extraction with prompt 10110 ...
Running extraction with prompt 10111 ...
Running extracti

In [21]:
# 3.2 Run batch extraction on the development dataset
results_development = {}
for prompt_id, prompt in prompts.items():
    print(f"Running extraction with prompt {prompt_id} ...")
    records = []

    development_output = batch_feature_extraction(model, prompt, development_data)
    development_output_path = os.path.join(extraction_dir, f'development_output_{prompt_id}.csv')
    development_output.to_csv(development_output_path, index=False) 
    
    results_development[prompt_id] = development_output

                     Study  Duration  Female    Age  SpeechDys  Seizures  \
0                      Xis  1.706917       1  27.00        NaN         1   
1       Odriozola-Grijalba  2.300292       0   4.00        1.0         1   
2  Agrawal - Wassmer, 2010  2.496178       1   1.83        1.0         1   
3     Novillo-López, Graus  1.732131       0  53.00        0.0         1   
4                Suri-Suri  2.262581       1  17.00        1.0         1   

   MvmtDis  Obtunded  BrainstemDys  BehCogImp  ...  IT2ndLineBort  \
0      1.0       NaN           NaN          1  ...              0   
1      1.0       1.0           0.0          1  ...              0   
2      1.0       1.0           1.0          1  ...              0   
3      0.0       0.0           0.0          1  ...              0   
4      1.0       1.0           1.0          1  ...              0   

   IT2ndLineToc  ITMaintenanceMMF  ITMaintenanceAZA  ITMaintenanceMTX  \
0             0                 0                 0    